<a href="https://colab.research.google.com/github/numan-art/Portfolio/blob/main/task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# ensure required packages are available; upgrade if necessary
%pip install -U pandas==2.2.2 scikit-learn nltk

import pandas as pd
import numpy as np
import re

# sklearn utilities – import after installation; fall back in case the
# linter/runtime still complains
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# persistence
import joblib

# nltk for stopwords/lemmatization
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [16]:
df = pd.read_csv('customer_support_tickets.csv')
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [17]:
df.info()

df['Ticket Type'].value_counts(dropna=False), df['Ticket Priority'].value_counts(dropna=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   object 
 2   Customer Email                8469 non-null   object 
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   object 
 5   Product Purchased             8469 non-null   object 
 6   Date of Purchase              8469 non-null   object 
 7   Ticket Type                   8469 non-null   object 
 8   Ticket Subject                8469 non-null   object 
 9   Ticket Description            8469 non-null   object 
 10  Ticket Status                 8469 non-null   object 
 11  Resolution                    2769 non-null   object 
 12  Ticket Priority               8469 non-null   object 
 13  Tic

(Ticket Type
 Refund request          1752
 Technical issue         1747
 Cancellation request    1695
 Product inquiry         1641
 Billing inquiry         1634
 Name: count, dtype: int64,
 Ticket Priority
 Medium      2192
 Critical    2129
 High        2085
 Low         2063
 Name: count, dtype: int64)

In [18]:
for col in ['Ticket Subject','Ticket Description']:
    df[col] = df[col].astype(str)

df['text'] = df['Ticket Subject'] + ' ' + df['Ticket Description']

In [19]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(s):
    s = s.lower()
    s = re.sub(r'[^a-z0-9\s]',' ', s)
    tokens = s.split()
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# apply cleaning

df['clean_text'] = df['text'].apply(clean_text)
df['clean_text'].head()

,clean_text
0,product setup issue product purchased please a...
1,peripheral compatibility issue product purchas...
2,network problem facing problem product purchas...
3,account access issue product purchased please ...
4,data loss issue product purchased please assis...


In [20]:
vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(df['clean_text'])
X.shape

(8469, 6258)

In [21]:
le_type = LabelEncoder()
le_priority = LabelEncoder()

y = le_type.fit_transform(df['Ticket Type'].fillna('Unknown'))
y_priority = le_priority.fit_transform(df['Ticket Priority'].fillna('Unknown'))

print('Categories:', list(le_type.classes_))
print('Priorities:', list(le_priority.classes_))

Categories: ['Billing inquiry', 'Cancellation request', 'Product inquiry', 'Refund request', 'Technical issue']
Priorities: ['Critical', 'High', 'Low', 'Medium']


In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
Xtr_pr, Xte_pr, ytr_pr, yte_pr = train_test_split(X, y_priority, test_size=0.2, random_state=42, stratify=y_priority)

In [ ]:
clf_cat_lr = LogisticRegression(max_iter=1000)
clf_cat_rf = RandomForestClassifier(n_estimators=100, random_state=42)

clf_cat_lr.fit(X_train, y_train)
clf_cat_rf.fit(X_train, y_train)

# priority models
clf_pr_lr = LogisticRegression(max_iter=1000)
clf_pr_rf = RandomForestClassifier(n_estimators=100, random_state=42)

clf_pr_lr.fit(Xtr_pr, ytr_pr)
clf_pr_rf.fit(Xtr_pr, ytr_pr)

In [ ]:
def evaluate(model, X_test, y_test, label_encoder, name):
    preds = model.predict(X_test)
    print(f"=== {name} ===")
    print("Accuracy:", accuracy_score(y_test, preds))
    print(classification_report(y_test, preds, target_names=label_encoder.classes_))
    print(confusion_matrix(y_test, preds))


print("Category predictions")
evaluate(clf_cat_lr, X_test, y_test, le_type, 'LR Category')
evaluate(clf_cat_rf, X_test, y_test, le_type, 'RF Category')

print("Priority predictions")
evaluate(clf_pr_lr, Xte_pr, yte_pr, le_priority, 'LR Priority')
evaluate(clf_pr_rf, Xte_pr, yte_pr, le_priority, 'RF Priority')

joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')
joblib.dump(clf_cat_rf, 'category_model.joblib')
joblib.dump(clf_pr_rf, 'priority_model.joblib')

# also save encoders
def save_encoder(enc, filename):
    joblib.dump(enc, filename)

save_encoder(le_type, 'labelenc_category.joblib')
save_encoder(le_priority, 'labelenc_priority.joblib')

print('Models and encoders saved.')